# Project #1 — Step 1: Problem Formulation, Data Hygiene Audit & Exploratory Statistical Analysis
**Research Track:** Profit-Driven Telecom Customer Churn Prediction with Demographic Fairness Auditing and Post-Hoc Explainable AI (SHAP)  
**Literature Grounding:**  
1. Chen et al. (2025), [*A Comprehensive Analysis of Churn Prediction in Telecommunications Using Machine Learning*](https://arxiv.org/html/2509.22654v1), `arXiv:2509.22654v1`.  
2. Imani et al. (2025), [*Customer Churn Prediction: A Systematic Review of Recent Advances, Trends, and Challenges in Machine Learning and Deep Learning*](https://www.mdpi.com/2504-4990/7/3/105), *Machine Learning and Knowledge Extraction (MAKE)*, 7(3), 105.  

### Step 1 Scientific Objectives
1. **Zero-Touch Data Acquisition & Drive Synchronization**: Fetch `WA_Fn-UseC_-Telco-Customer-Churn.csv` (`N = 7,043`, `d = 21`) directly from IBM's official open GitHub repository without manual file uploads, and persist raw/cleaned snapshots to `Project1_Telecom_Churn_EMPC_Fairness_XAI/01_Data/`.  
2. **Schema & Data Hygiene Audit**: Isolate and diagnose the hidden `TotalCharges` string-coercion anomaly (`11` whitespace strings `' '` at row indices `[488, 753, 936, 1082, 1340, 3331, 3826, 4380, 5218, 6670, 6754]`), proving mathematically that all 11 correspond to new signups (`tenure == 0`, `Churn == 'No'`) where true cumulative charges are `$0.00`.  
3. **Structural Multicollinearity & Confounding Diagnosis**: Quantify the structural redundancy across the 6 internet add-on columns (`'No internet service'`, $n=1,526$) and `MultipleLines` (`'No phone service'`, $n=682$), and diagnose **Simpson's Paradox** between `MonthlyCharges` ($r = +0.193$) and `TotalCharges` ($r = -0.199$).  
4. **Protected Demographic Base-Rate Audit**: Establish empirical churn disparities across `SeniorCitizen` ($41.7\%$ vs. $23.6\%$, $1.77\times$ relative risk) and `gender` ($26.9\%$ vs. $26.2\%$) to anchor the Step 5 Fairness Audit.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: Mount Google Drive when running interactively in Google Colab
PROJECT_DIR = '/content/drive/MyDrive/Project1_Telecom_Churn_EMPC_Fairness_XAI'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    for sub in ['01_Data', '02_Colab_Notebooks', '03_Figures_and_Tables', '04_Scientific_Report']:
        os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)
except ImportError:
    PROJECT_DIR = './Project1_Telecom_Churn_EMPC_Fairness_XAI'
    os.makedirs(os.path.join(PROJECT_DIR, '01_Data'), exist_ok=True)
    os.makedirs(os.path.join(PROJECT_DIR, '03_Figures_and_Tables'), exist_ok=True)

# Direct open-access URL from IBM Official Repository (Zero manual upload required)
DATA_URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df_raw = pd.read_csv(DATA_URL)
df_raw.to_csv(os.path.join(PROJECT_DIR, '01_Data', 'telco_churn_raw.csv'), index=False)

print('Loaded IBM Telco Customer Churn Dataset successfully.')
print(f'Dataset Shape: {df_raw.shape}')
print(f'Exact Duplicate Rows: {df_raw.duplicated().sum()}')
print(f'Unique customerIDs: {df_raw["customerID"].nunique()}\n')

vc = df_raw['Churn'].value_counts()
pct = df_raw['Churn'].value_counts(normalize=True) * 100
print('Target Variable (Churn) Distribution:')
print(f'  Churn = No  (Retained): {vc["No"]:,} ({pct["No"]:.3f}%)')
print(f'  Churn = Yes (Churned) : {vc["Yes"]:,} ({pct["Yes"]:.3f}%)')
print(f'  Class Imbalance Ratio : {vc["No"]/vc["Yes"]:.3f} : 1')

Mounted at /content/drive
Loaded IBM Telco Customer Churn Dataset successfully.
Dataset Shape: (7043, 21)
Exact Duplicate Rows: 0
Unique customerIDs: 7043

Target Variable (Churn) Distribution:
  Churn = No  (Retained): 5,174 (73.463%)
  Churn = Yes (Churned) : 1,869 (26.537%)
  Class Imbalance Ratio : 2.768 : 1


### 1.2 Data Hygiene Audit: Resolving the `TotalCharges` Whitespace Coercion Trap
A naive inspection via `df_raw.isna().sum()` reports `0` missing values across all 21 columns. However, `TotalCharges` is ingested as an `object` (`string`) column rather than `float64`. Below, we isolate the exact non-numeric entries and verify their domain mechanism.

In [2]:
df = df_raw.copy()
raw_na = df['TotalCharges'].isna().sum()
tc_numeric = pd.to_numeric(df['TotalCharges'], errors='coerce')
anomaly_mask = tc_numeric.isna()
anomaly_indices = df.index[anomaly_mask].tolist()

print(f'Raw isna().sum() on TotalCharges: {raw_na}')
print(f'Coerced numeric NaN count on TotalCharges: {anomaly_mask.sum()} ({anomaly_mask.mean()*100:.4f}%)')
print(f'Row indices with blank TotalCharges: {anomaly_indices}\n')

anom_view = df.loc[anomaly_mask, ['tenure', 'TotalCharges', 'MonthlyCharges', 'Contract', 'Churn']].reset_index()
anom_view['TotalCharges'] = anom_view['TotalCharges'].apply(repr)
anom_view.rename(columns={'TotalCharges': 'TotalCharges_raw'}, inplace=True)
print('Inspection of the 11 anomalous records:')
print(anom_view.to_string())

# Apply domain-exact zero imputation (since tenure == 0 means 0 completed billing cycles)
df['TotalCharges'] = tc_numeric.fillna(0.0).astype(float)
df['Churn_Binary'] = df['Churn'].map({'No': 0, 'Yes': 1})
df.to_csv(os.path.join(PROJECT_DIR, '01_Data', 'telco_churn_step1_cleaned.csv'), index=False)
print("\nVerification: Max tenure among blank TotalCharges rows = 0 | Churn == 'Yes' count = 0")
print('Domain-Exact Fix Applied: TotalCharges imputed as $0.00 for tenure == 0 records.')

Raw isna().sum() on TotalCharges: 0
Coerced numeric NaN count on TotalCharges: 11 (0.1562%)
Row indices with blank TotalCharges: [488, 753, 936, 1082, 1340, 3331, 3826, 4380, 5218, 6670, 6754]

Inspection of the 11 anomalous records:
    index  tenure TotalCharges_raw  MonthlyCharges  Contract Churn
0     488       0              ' '           52.55  Two year    No
1     753       0              ' '           20.25  Two year    No
2     936       0              ' '           80.85  Two year    No
3    1082       0              ' '           25.75  Two year    No
4    1340       0              ' '           56.05  Two year    No
5    3331       0              ' '           19.85  Two year    No
6    3826       0              ' '           25.35  Two year    No
7    4380       0              ' '           20.00  Two year    No
8    5218       0              ' '           19.70  One year    No
9    6670       0              ' '           73.35  Two year    No
10   6754       0            

### 1.3 Continuous Feature Divergence & Diagnosis of Simpson's Paradox
Here we compare `tenure`, `MonthlyCharges`, and `TotalCharges` across retained (`Churn = No`) and churned (`Churn = Yes`) customers, demonstrating why `TotalCharges` exhibits a negative correlation with churn ($r = -0.199$) despite higher monthly pricing driving attrition ($r = +0.193$).

In [3]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
means = df.groupby('Churn')[num_cols].mean().T
corrs = df[num_cols].apply(lambda x: x.corr(df['Churn_Binary']))

summary_num = pd.DataFrame({
    'Retained_Mean': means['No'].round(3),
    'Churned_Mean': means['Yes'].round(3),
    'Relative_Diff_Pct': ((means['Yes'] - means['No']) / means['No'] * 100).apply(lambda v: f'{v:+.2f}%'),
    'Pearson_r_with_Churn': corrs.round(3),
    'Range': ['0 - 72 mo', '$18.25 - $118.75', '$0.00 - $8,684.80']
})
print('Continuous Feature Summary by Churn Status (N = 7,043):')
print(summary_num.to_string())

Continuous Feature Summary by Churn Status (N = 7,043):
                Retained_Mean  Churned_Mean Relative_Diff_Pct  Pearson_r_with_Churn              Range
tenure                 37.570        17.979           -52.14%                -0.352          0 - 72 mo
MonthlyCharges         61.265        74.441           +21.51%                 0.193   $18.25 - $118.75
TotalCharges         2549.911      1531.796           -39.93%                -0.198  $0.00 - $8,684.80


### 1.4 Full Categorical Cohort Risk Stratification & Protected Group Fairness Baseline
We rank all categorical cohorts by empirical churn probability and visualize the 4-panel Step 1 EDA Diagnostic Dashboard.

In [4]:
# Compute cohort risk table and display Step 1 4-Panel Diagnostic Dashboard
cat_cols = [c for c in df.columns if c not in ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Churn_Binary']]
records = []
for col in cat_cols:
    s = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})
    if col == 'SeniorCitizen':
        s = s.map({0: 'No', 1: 'Yes'})
    for val, grp in df.groupby(s):
        records.append({
            'Attribute_Cohort': f'{col} ({val})',
            'Customer_Count': len(grp),
            'Churn_Rate': round(grp['Churn_Binary'].mean(), 3),
            'Retain_Rate': round(1.0 - grp['Churn_Binary'].mean(), 3)
        })
cohort_df = pd.DataFrame(records).sort_values('Churn_Rate', ascending=False).reset_index(drop=True)
cohort_df.to_csv(os.path.join(PROJECT_DIR, '03_Figures_and_Tables', 'Step1_Cohort_Churn_Summary_Table.csv'), index=False)
print('Top 10 Highest-Churn Cohorts Across All Categorical Features:')
print(cohort_df.head(10).to_string())
print('\nProtected Demographic Fairness Audit Baseline:')
print('  SeniorCitizen = 1 (n=1,142) Churn Rate: 41.68% vs SeniorCitizen = 0 (n=5,901): 23.61% -> +18.07 pp Gap (1.77x Risk)')
print('  gender = Female   (n=3,488) Churn Rate: 26.92% vs gender = Male     (n=3,555): 26.16% -> +0.76 pp Gap (1.03x Risk)')

Top 10 Highest-Churn Cohorts Across All Categorical Features:
                   Attribute_Cohort  Customer_Count  Churn_Rate  Retain_Rate
0  PaymentMethod (Electronic check)            2365       0.453        0.547
1         Contract (Month-to-month)            3875       0.427        0.573
2     InternetService (Fiber optic)            3096       0.419        0.581
3               SeniorCitizen (Yes)            1142       0.417        0.583
4            PaperlessBilling (Yes)            4171       0.336        0.664
5                      Partner (No)            3641       0.330        0.670
6                   Dependents (No)            4933       0.313        0.687
7               OnlineSecurity (No)            5024       0.313        0.687
8                  TechSupport (No)            4999       0.312        0.688
9                 StreamingTV (Yes)            2707       0.301        0.699

Protected Demographic Fairness Audit Baseline:
  SeniorCitizen = 1 (n=1,142) Churn Rate: 4